# HANS Evaluation

Evaluate an MNLI fine tuned BERT model on HANS to analyze performance across lexical overlap, subsequence and constituent heuristics.

## Imports

In [1]:
import torch
import pandas as pandas
import numpy as np

from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer

## Configuration

In [ ]:
MAX_LENGTH = 128
SEED = 456

MODEL_PATH = f"../models/fp32/seed{SEED}/final"

print("Model checkpoint:", MODEL_PATH)
print("Max sequence length:", MAX_LENGTH)
print("Random seed:", SEED)

Model checkpoint: ../models/fp32/seed456/final
Max sequence length: 128
Random seed: 456


In [3]:
import os

print("Working directory:", os.getcwd())
print("Model exists:", os.path.exists(MODEL_PATH))
print("Checkpoint files:", os.listdir(MODEL_PATH))

Working directory: /Users/tejaspramesh/Desktop/Quantization-Spurious-Heuristics-in-Transformers/notebooks
Model exists: True
Checkpoint files: ['model.safetensors', 'tokenizer_config.json', 'config.json', 'tokenizer.json', 'training_args.bin']


## LOAD MNLI Fine Tuned Model

In [4]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, local_files_only=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

print("ID to label:", model.config.id2label)
print("Label to ID:", model.config.label2id)
print("Tokenizer:", tokenizer.name_or_path)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ID to label: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
Label to ID: {'contradiction': 2, 'entailment': 0, 'neutral': 1}
Tokenizer: ../models/fp32/seed456/final


## Load HANS

In [5]:
HANS_URL = (
    "https://raw.githubusercontent.com/tommccoy1/hans/"
    "master/heuristics_evaluation_set.txt"
)

hans = load_dataset(
    "csv",
    data_files={"validation": HANS_URL},
    delimiter="\t"
)

hans

DatasetDict({
    validation: Dataset({
        features: ['gold_label', 'sentence1_binary_parse', 'sentence2_binary_parse', 'sentence1_parse', 'sentence2_parse', 'sentence1', 'sentence2', 'pairID', 'heuristic', 'subcase', 'template'],
        num_rows: 30000
    })
})

In [6]:
hans_eval = hans["validation"]
print(f"Number of HANS examples: {len(hans_eval)}")
print(hans_eval.column_names)

Number of HANS examples: 30000
['gold_label', 'sentence1_binary_parse', 'sentence2_binary_parse', 'sentence1_parse', 'sentence2_parse', 'sentence1', 'sentence2', 'pairID', 'heuristic', 'subcase', 'template']


## Prepare HANS Eval Data

In [7]:
columns_to_keep = ["gold_label", "heuristic", "subcase", "template", "pairID", "sentence1", "sentence2"]

columns_to_remove = [
    column for column in hans_eval.column_names if column not in columns_to_keep
]

hans_eval = hans_eval.remove_columns(columns_to_remove)
hans_eval

Dataset({
    features: ['gold_label', 'sentence1', 'sentence2', 'pairID', 'heuristic', 'subcase', 'template'],
    num_rows: 30000
})

In [8]:
hans_eval = hans_eval.rename_columns({"sentence1": "premise", "sentence2": "hypothesis"})
print(hans_eval.column_names)


['gold_label', 'premise', 'hypothesis', 'pairID', 'heuristic', 'subcase', 'template']


In [9]:
#Inspecting

example = hans_eval[0]
print("Premise:", example["premise"])
print("Hypothesis:", example["hypothesis"])
print("Gold label:", example["gold_label"])
print("Heuristic:", example["heuristic"])
print("Subcase:", example["subcase"])

Premise: The president advised the doctor .
Hypothesis: The doctor advised the president .
Gold label: non-entailment
Heuristic: lexical_overlap
Subcase: ln_subject/object_swap


## Load Tokenizer

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
print("Tokenizer:", tokenizer.name_or_path)

Tokenizer: ../models/fp32/seed456/final


## Tokenize HANS

In [11]:
def tokenize_function(batch):
    return tokenizer(
        batch["premise"],
        batch["hypothesis"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_hans = hans_eval.map(tokenize_function, batched=True)
tokenized_hans

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Dataset({
    features: ['gold_label', 'premise', 'hypothesis', 'pairID', 'heuristic', 'subcase', 'template', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 30000
})

In [12]:
# Check, Inspect Tokenization

print("Premise:")
print(tokenized_hans[0]["premise"])

print("\nHypothesis:")
print(tokenized_hans[0]["hypothesis"])

print("\nToken IDs:")
print(tokenized_hans[0]["input_ids"])

print("\nDecoded:")
print(tokenizer.decode(tokenized_hans[0]["input_ids"]))

Premise:
The president advised the doctor .

Hypothesis:
The doctor advised the president .

Token IDs:
[101, 1996, 2343, 9449, 1996, 3460, 1012, 102, 1996, 3460, 9449, 1996, 2343, 1012, 102]

Decoded:
[CLS] the president advised the doctor. [SEP] the doctor advised the president. [SEP]


## Set Evaluation Device

In [13]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device)
model.eval()

print("Evaluation Device:", device)

Evaluation Device: mps


## Prepare HANS DataLoader

In [14]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer,return_tensors="pt")

model_columns = ["input_ids", "attention_mask", "token_type_ids"]
hans_model_inputs = tokenized_hans.select_columns(model_columns)

hans_dataloaders = DataLoader(
    hans_model_inputs,
    batch_size=32,
    shuffle=False,
    collate_fn=data_collator,
)

print("Number of batches:", len(hans_dataloaders))
print("Number of examples:", len(hans_model_inputs))

Number of batches: 938
Number of examples: 30000


In [15]:
sample_batch = next(iter(hans_dataloaders))
print("Batch keys:", sample_batch.keys())
print("Batch input_ids shape:", sample_batch["input_ids"].shape)
print("Batch attention_mask shape:", sample_batch["attention_mask"].shape)
print("Batch token_type_ids shape:", sample_batch["token_type_ids"].shape)

Batch keys: KeysView({'input_ids': tensor([[  101,  1996,  2343,  9449,  1996,  3460,  1012,   102,  1996,  3460,
          9449,  1996,  2343,  1012,   102],
        [  101,  1996,  3076,  2387,  1996, 10489,  1012,   102,  1996, 10489,
          2387,  1996,  3076,  1012,   102],
        [  101,  1996, 11274,  6628,  1996, 13448,  1012,   102,  1996, 13448,
          6628,  1996, 11274,  1012,   102],
        [  101,  1996, 10153,  3569,  1996,  3364,  1012,   102,  1996,  3364,
          3569,  1996, 10153,  1012,   102],
        [  101,  1996,  5889,  9511,  1996, 25375,  1012,   102,  1996, 25375,
          9511,  1996,  5889,  1012,   102],
        [  101,  1996, 10153,  3855,  1996,  3063,  1012,   102,  1996,  3063,
          3855,  1996, 10153,  1012,   102],
        [  101,  1996, 10489,  2387,  1996, 23660,  1012,   102,  1996, 23660,
          2387,  1996, 10489,  1012,   102],
        [  101,  1996,  2934,  3858,  1996, 23660,  1012,   102,  1996, 23660,
          3858,  1

## Run FP32 Inference

In [16]:
all_logits = []
all_probabilities = []
all_mnli_predictions = []

with torch.no_grad():
    for batch in hans_dataloaders:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=-1)
        predictions = torch.argmax(probabilities, dim=-1)

        all_logits.append(logits.cpu())
        all_probabilities.append(probabilities.cpu())
        all_mnli_predictions.append(predictions.cpu())
        


In [17]:
all_logits = torch.cat(all_logits)
all_probabilities = torch.cat(all_probabilities)
all_mnli_predictions = torch.cat(all_mnli_predictions)

print("All logits shape:", all_logits.shape)
print("All probabilities shape:", all_probabilities.shape)
print("All MNLI predictions shape:", all_mnli_predictions.shape)

All logits shape: torch.Size([30000, 3])
All probabilities shape: torch.Size([30000, 3])
All MNLI predictions shape: torch.Size([30000])


## Map MLNI Predictions to HANS Labels

In [18]:
def mnli_to_hans_label(prediction):
    if prediction == 0:
        return "entailment"
    return "non-entailment"

hans_predictions = [mnli_to_hans_label(pred.item()) for pred in all_mnli_predictions]

print("First 10 HANS predictions:", hans_predictions[:10])
print("Number of HANS predictions:", len(hans_predictions))

First 10 HANS predictions: ['entailment', 'entailment', 'entailment', 'entailment', 'entailment', 'entailment', 'entailment', 'entailment', 'entailment', 'entailment']
Number of HANS predictions: 30000


## Create Prediction Results

In [19]:
results_df = hans_eval.to_pandas()
results_df["mnli_prediction_id"] = all_mnli_predictions.numpy()
results_df["mnli_prediction"] = [model.config.id2label[pred] for pred in results_df["mnli_prediction_id"]]

results_df["hans_prediction"] = hans_predictions
results_df["correct"] = (
    results_df["hans_prediction"] == results_df["gold_label"]
    )

results_df.head()

,gold_label,premise,hypothesis,pairID,heuristic,subcase,template,mnli_prediction_id,mnli_prediction,hans_prediction,correct
0,non-entailment,The president advised the doctor .,The doctor advised the president .,ex0,lexical_overlap,ln_subject/object_swap,temp1,0,entailment,entailment,False
1,non-entailment,The student saw the managers .,The managers saw the student .,ex1,lexical_overlap,ln_subject/object_swap,temp1,0,entailment,entailment,False
2,non-entailment,The presidents encouraged the banker .,The banker encouraged the presidents .,ex2,lexical_overlap,ln_subject/object_swap,temp1,0,entailment,entailment,False
3,non-entailment,The senators supported the actor .,The actor supported the senators .,ex3,lexical_overlap,ln_subject/object_swap,temp1,0,entailment,entailment,False
4,non-entailment,The actors avoided the bankers .,The bankers avoided the actors .,ex4,lexical_overlap,ln_subject/object_swap,temp1,0,entailment,entailment,False


## Overall HANS Accuracy

In [20]:
overall_accuracy = results_df["correct"].mean()
print(f"Overall accuracy on HANS dataset: {overall_accuracy:.4f}")
print(f"Overall accuracy on HANS dataset: {overall_accuracy*100:.2f}%")

Overall accuracy on HANS dataset: 0.5385
Overall accuracy on HANS dataset: 53.85%


## Accuracy by HANS Label

In [21]:
label_results = (results_df.groupby("gold_label")["correct"].agg(["mean", "count"]))

label_results["accuracy_percentage"] = label_results["mean"] * 100
label_results

,mean,count,accuracy_percentage
gold_label,,,
entailment,0.989933,15000,98.993333
non-entailment,0.087000,15000,8.700000


## Accuracy by HANS Heuristic

In [22]:
heuristic_results = (results_df.groupby("heuristic")["correct"].agg(["mean", "count"]))
heuristic_results["accuracy_percentage"] = (heuristic_results["mean"] * 100)
heuristic_results

,mean,count,accuracy_percentage
heuristic,,,
constituent,0.5234,10000,52.34
lexical_overlap,0.5822,10000,58.22
subsequence,0.5098,10000,50.98


## Accuracy by Heuristic and Label

In [23]:
heuristic_label_results = (results_df.groupby(["heuristic", "gold_label"])["correct"].agg(["mean", "count"]))
heuristic_label_results["accuracy_percentage"] = (heuristic_label_results["mean"] * 100)
heuristic_label_results

mean  count  accuracy_percentage
heuristic       gold_label                                        
constituent     entailment      0.9994   5000                99.94
                non-entailment  0.0474   5000                 4.74
lexical_overlap entailment      0.9746   5000                97.46
                non-entailment  0.1898   5000                18.98
subsequence     entailment      0.9958   5000                99.58
                non-entailment  0.0238   5000                 2.38

## Accuracy by HANS Subcase

In [24]:
subcase_results = (results_df.groupby(["heuristic", "gold_label" , "subcase"])["correct"].agg(["mean", "count"]))
subcase_results["accuracy_percentage"] = (subcase_results["mean"] * 100)
subcase_results


mean  count  \
heuristic       gold_label     subcase                                        
constituent     entailment     ce_adverb                       1.000   1000   
                               ce_after_since_clause           1.000   1000   
                               ce_conjunction                  1.000   1000   
                               ce_embedded_under_since         0.998   1000   
                               ce_embedded_under_verb          0.999   1000   
                non-entailment cn_adverb                       0.012   1000   
                               cn_after_if_clause              0.000   1000   
                               cn_disjunction                  0.001   1000   
                               cn_embedded_under_if            0.167   1000   
                               cn_embedded_under_verb          0.057   1000   
lexical_overlap entailment     le_around_prepositional_phrase  1.000   1000   
                               le_around_relative_clause       0.994   1000   
                               le_conjunction                  0.885   1000   
                               le_passive                      1.000   1000   
                               le_relative_clause              0.994   1000   
                non-entailment ln_conjunction                  0.344   1000   
                               ln_passive                      0.000   1000   
                               ln_preposition                  0.321   1000   
                               ln_relative_clause              0.241   1000   
                               ln_subject/object_swap          0.043   1000   
subsequence     entailment     se_PP_on_obj                    1.000   1000   
                               se_adjective                    1.000   1000   
                               se_conjunction                  0.979   1000   
                               se_relative_clause_on_obj       1.000   1000   
                               se_understood_object            1.000   1000   
                non-entailment sn_NP/S                         0.004   1000   
                               sn_NP/Z                         0.011   1000   
                               sn_PP_on_subject                0.060   1000   
                               sn_past_participle              0.001   1000   
                               sn_relative_clause_on_subject   0.043   1000   

                                                               accuracy_percentage  
heuristic       gold_label     subcase                                              
constituent     entailment     ce_adverb                                     100.0  
                               ce_after_since_clause                         100.0  
                               ce_conjunction                                100.0  
                               ce_embedded_under_since                        99.8  
                               ce_embedded_under_verb                         99.9  
                non-entailment cn_adverb                                       1.2  
                               cn_after_if_clause                              0.0  
                               cn_disjunction                                  0.1  
                               cn_embedded_under_if                           16.7  
                               cn_embedded_under_verb                          5.7  
lexical_overlap entailment     le_around_prepositional_phrase                100.0  
                               le_around_relative_clause                      99.4  
                               le_conjunction                                 88.5  
                               le_passive                                    100.0  
                               le_relative_clause                             99.4  
                non-entailment ln_conjunction                                 34.4  
     

## Save FP32 HANS Predictions

In [25]:
results_df["prob_entailment"] = all_probabilities[:, 0].numpy()
results_df["prob_neutral"] = all_probabilities[:, 1].numpy()
results_df["prob_contradiction"] = all_probabilities[:, 2].numpy()

results_df["prob_non_entailment"] = (
    results_df["prob_neutral"] + results_df["prob_contradiction"]
)

results_df.head()

,gold_label,premise,hypothesis,pairID,heuristic,subcase,template,mnli_prediction_id,mnli_prediction,hans_prediction,correct,prob_entailment,prob_neutral,prob_contradiction,prob_non_entailment
0,non-entailment,The president advised the doctor .,The doctor advised the president .,ex0,lexical_overlap,ln_subject/object_swap,temp1,0,entailment,entailment,False,0.970154,0.004267,0.025579,0.029846
1,non-entailment,The student saw the managers .,The managers saw the student .,ex1,lexical_overlap,ln_subject/object_swap,temp1,0,entailment,entailment,False,0.979043,0.007708,0.013249,0.020957
2,non-entailment,The presidents encouraged the banker .,The banker encouraged the presidents .,ex2,lexical_overlap,ln_subject/object_swap,temp1,0,entailment,entailment,False,0.958382,0.011345,0.030273,0.041618
3,non-entailment,The senators supported the actor .,The actor supported the senators .,ex3,lexical_overlap,ln_subject/object_swap,temp1,0,entailment,entailment,False,0.843129,0.032325,0.124545,0.156871
4,non-entailment,The actors avoided the bankers .,The bankers avoided the actors .,ex4,lexical_overlap,ln_subject/object_swap,temp1,0,entailment,entailment,False,0.921192,0.020876,0.057932,0.078808


In [26]:
from pathlib import Path

RESULTS_DIR = Path("../results/hans")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

output_path = RESULTS_DIR / f"fp32_seed{SEED}_predictions.csv"

results_df.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Number of rows:", len(results_df))

Saved to: ../results/hans/fp32_seed456_predictions.csv
Number of rows: 30000


In [27]:
saved_results = pandas.read_csv(output_path)

print("Saved rows:", len(saved_results))
print("Saved columns:", saved_results.columns.tolist())

Saved rows: 30000
Saved columns: ['gold_label', 'premise', 'hypothesis', 'pairID', 'heuristic', 'subcase', 'template', 'mnli_prediction_id', 'mnli_prediction', 'hans_prediction', 'correct', 'prob_entailment', 'prob_neutral', 'prob_contradiction', 'prob_non_entailment']


## Save FP32 HANS Summary

In [28]:
import json

summary = {
    "model": "google-bert/bert-base-uncased",
    "seed": SEED,
    "dataset": "HANS",
    "num_examples": len(results_df),

    "overall_accuracy": float(results_df["correct"].mean()),

    "by_label": {
        label: float(group["correct"].mean())
        for label, group in results_df.groupby("gold_label")
    },

    "by_heuristic": {
        heuristic: float(group["correct"].mean())
        for heuristic, group in results_df.groupby("heuristic")
    },

    "by_heuristic_and_label": {
        heuristic: {
            label: float(label_group["correct"].mean())
            for label, label_group
            in heuristic_group.groupby("gold_label")
        }
        for heuristic, heuristic_group
        in results_df.groupby("heuristic")
    }
}

summary

{'model': 'google-bert/bert-base-uncased',
 'seed': 456,
 'dataset': 'HANS',
 'num_examples': 30000,
 'overall_accuracy': 0.5384666666666666,
 'by_label': {'entailment': 0.9899333333333333, 'non-entailment': 0.087},
 'by_heuristic': {'constituent': 0.5234,
  'lexical_overlap': 0.5822,
  'subsequence': 0.5098},
 'by_heuristic_and_label': {'constituent': {'entailment': 0.9994,
   'non-entailment': 0.0474},
  'lexical_overlap': {'entailment': 0.9746, 'non-entailment': 0.1898},
  'subsequence': {'entailment': 0.9958, 'non-entailment': 0.0238}}}

In [29]:
summary_path = RESULTS_DIR / f"fp32_seed{SEED}_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=4)

print("Saved to:", summary_path)

Saved to: ../results/hans/fp32_seed456_summary.json


In [30]:
with open(summary_path, "r") as f:
    saved_summary = json.load(f)

print(json.dumps(saved_summary, indent=4))

{
    "model": "google-bert/bert-base-uncased",
    "seed": 456,
    "dataset": "HANS",
    "num_examples": 30000,
    "overall_accuracy": 0.5384666666666666,
    "by_label": {
        "entailment": 0.9899333333333333,
        "non-entailment": 0.087
    },
    "by_heuristic": {
        "constituent": 0.5234,
        "lexical_overlap": 0.5822,
        "subsequence": 0.5098
    },
    "by_heuristic_and_label": {
        "constituent": {
            "entailment": 0.9994,
            "non-entailment": 0.0474
        },
        "lexical_overlap": {
            "entailment": 0.9746,
            "non-entailment": 0.1898
        },
        "subsequence": {
            "entailment": 0.9958,
            "non-entailment": 0.0238
        }
    }
}
